In [ ]:
!pip install -q gensim pandas numpy

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import gensim
from gensim.models import Word2Vec, FastText

print(f"Gensim version: {gensim.__version__}")

Gensim version: 4.4.0


In [2]:
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/nlp'):
        !git clone -b lab-09-branch --single-branch https://github.com/jaYulichka46/nlp.git
    
    %cd /content/nlp
    !pip install pandas numpy scikit-learn matplotlib seaborn -q
    sys.path.append('/content/nlp')

    FOLDER_ID = '1Xhu4xjZpRu-RP730-hyErp5F0C3l_EvO'
    
    os.makedirs('data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/
    
    data_dir = 'data/processed_v2'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data/processed_v2'

In [3]:
### 2. Data access
if 'google.colab' in sys.modules:
    data_file = os.path.join(data_dir, 'sample.csv')
else:
    data_file = os.path.join(data_dir, 'processed_v_2.csv')

df = pd.read_csv(data_file, nrows=5000)

text_col = 'processed_text' if 'processed_text' in df.columns else 'clean_text'

df = df.dropna(subset=[text_col])

print(f"Успішно завантажено {len(df)} рядків. Робоча колонка: '{text_col}'")

Успішно завантажено 5000 рядків. Робоча колонка: 'processed_text'


### Preprocessing & Tokenization for Embeddings (UA Datasets)

In [4]:
# 1. Відкидаємо короткі документи
df['word_count'] = df[text_col].apply(lambda x: len(str(x).split()))
df_filtered = df[df['word_count'] >= 20].copy()

# 2. Токенізація
tokenized_corpus = [str(text).split() for text in df_filtered[text_col]]

# 3. Формування звіту
total_docs = len(tokenized_corpus)
total_tokens = sum(len(doc) for doc in tokenized_corpus)

print(f"Тип тексту: '{text_col}' (слова у вихідній формі без лематизації).")
print(f"Обґрунтування: Українська мова морфологічно багата. Лематизація знищить варіативність відмінків, і ми не зможемо перевірити ефективність FastText на підсловних n-грамах.")
print(f"Метод токенізації: поділ по пробілах (str.split()), оскільки текст вже очищено від пунктуації.")
print(f"Документів ДО фільтрації: {len(df)}")
print(f"Документів ПІСЛЯ фільтрації (>= 20 слів): {total_docs}")
print(f"Загальна кількість токенів (слів): {total_tokens:,}")

Тип тексту: 'processed_text' (слова у вихідній формі без лематизації).
Обґрунтування: Українська мова морфологічно багата. Лематизація знищить варіативність відмінків, і ми не зможемо перевірити ефективність FastText на підсловних n-грамах.
Метод токенізації: поділ по пробілах (str.split()), оскільки текст вже очищено від пунктуації.
Документів ДО фільтрації: 5000
Документів ПІСЛЯ фільтрації (>= 20 слів): 4939
Загальна кількість токенів (слів): 965,294


### Training Word Embeddings (Word2Vec & FastText)

In [ ]:
import multiprocessing

# Визначаємо кількість ядер процесора для максимального прискорення
WORKERS = multiprocessing.cpu_count() - 1 
if WORKERS < 1: WORKERS = 1

# Однакові параметри для чесного порівняння
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 5  # Підняли до 5 через величезний об'єм даних (23 млн слів)
SG = 1         # 1 = Skip-gram, 0 = CBOW

print("Параметри тренування:")
print(f"vector_size={VECTOR_SIZE}")
print(f"window={WINDOW}")
print(f"min_count={MIN_COUNT} (відкидаємо слова, що зустрічаються рідше)")
print(f"sg={SG} (Skip-gram)")
print(f"workers={WORKERS} (потоки процесора)")

Параметри тренування:
vector_size=100
window=5
min_count=5 (відкидаємо слова, що зустрічаються рідше)
sg=1 (Skip-gram)
workers=11 (потоки процесора)


In [6]:
# 1. Тренування Word2Vec
start_time = time.time()
w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=SG,
    workers=WORKERS
)
print(f"Word2Vec натреновано за {time.time() - start_time:.2f} сек.")
print(f"Розмір словника Word2Vec: {len(w2v_model.wv.key_to_index):,} унікальних слів")

Word2Vec натреновано за 4.65 сек.
Розмір словника Word2Vec: 23,580 унікальних слів


In [7]:
# 2. Тренування FastText
start_time = time.time()
ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=SG,
    workers=WORKERS
)
print(f"FastText натреновано за {time.time() - start_time:.2f} сек.")
print(f"Розмір словника FastText: {len(ft_model.wv.key_to_index):,} унікальних слів")

FastText натреновано за 12.02 сек.
Розмір словника FastText: 23,580 унікальних слів


### Обґрунтування вибору параметрів (Пункт 2.2)

Для забезпечення **абсолютно чесного порівняння** ми тренуємо Word2Vec та FastText з ідентичними налаштуваннями на нашій репрезентативній вибірці з датасету (~965 тис. токенів).

* `vector_size=100`: Оптимальна розмірність векторів. Достатня для захоплення семантики, при цьому ідеально підходить для корпусу розміром близько 1 млн слів (уникаємо перенавчання).
* `window=5`: Стандартне вікно, яке добре вловлює локальний контекст у реченнях українською мовою.
* `min_count=5`: Оскільки наш словник досить великий, поріг у 5 згадувань є оптимальним фільтром. Слово, яке зустрічається рідше 5 разів на мільйон слів, є статистичним шумом або рідкісною одруківкою, яка лише заважатиме Word2Vec (але FastText все одно зможе з нею працювати завдяки n-грамам).
* `sg=1` (Skip-gram): **Ключовий вибір для української мови**. CBOW (`sg=0`) усереднює контекст і працює швидше, але Skip-gram значно краще моделює семантику **рідкісних слів та словоформ**. Оскільки українська мова є флективною (має багато відмінків, префіксів та суфіксів), багато форм одного слова зустрічаються не так часто. Skip-gram дозволяє отримати набагато якісніші вектори для таких специфічних словоформ.

### Nearest Neighbors Analysis (10 words)

In [8]:
# 10 спеціально підібраних слів для українського корпусу
test_words = {
    "держава": "Часте слово (General)",
    "року": "Часте слово (General)",
    "стартап": "Рідкісне/Специфічне доменне слово",
    "бюджет": "Доменний термін (News/Economics)",
    "інвестиції": "Доменний термін (News/Economics)",
    "місто": "Морфологічна варіативність (Називний відм.)",
    "міста": "Морфологічна варіативність (Родовий відм.)",
    "проєкт": "Варіативність правопису / Новий правопис",
    "онлайн": "Запозичення / Трансліт",
    "презедент": "Навмисна опечатка (немає 'и')"
}

results = []

In [ ]:
def get_top_5(model, word, is_fasttext=False):
    try:
        # Для FastText most_similar працює навіть для OOV (Out of Vocabulary)
        if is_fasttext or word in model.wv.key_to_index:
            neighbors = model.wv.most_similar(word, topn=5)
            return ", ".join([f"{w} ({sim:.2f})" for w, sim in neighbors])
        else:
            return "немає в словнику"
    except KeyError:
        return "немає в словнику"

In [10]:
for word, category in test_words.items():
    w2v_res = get_top_5(w2v_model, word, is_fasttext=False)
    ft_res = get_top_5(ft_model, word, is_fasttext=True)
    
    results.append({
        "Категорія": category,
        "Слово": word,
        "Word2Vec (Топ-5)": w2v_res,
        "FastText (Топ-5)": ft_res
    })

df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(df_results)

,Категорія,Слово,Word2Vec (Топ-5),FastText (Топ-5)
0,Часте слово (General),держава,"зберегти (0.93), відбуватися (0.93), програма (0.92), криза (0.92), додати (0.92)","держава"" (0.97), державу (0.96), державою. (0.95), державі, (0.94), державі (0.94)"
1,Часте слово (General),року,"року. (0.85), року, (0.77), році (0.75), тижня (0.73), березні (0.72)","року? (0.96), року) (0.96), року.У (0.94), року». (0.92), року). (0.91)"
2,Рідкісне/Специфічне доменне слово,стартап,"технологічний (0.98), бібліотека (0.97), Роберта (0.97), визнають, (0.97), рід (0.97)","стартапи (0.99), старт (0.96), стартує (0.95), старті (0.93), старту (0.91)"
3,Доменний термін (News/Economics),бюджет,"дефіцит (0.94), прибуток (0.93), перевищили (0.92), рік, (0.92), 3% (0.92)","бюджет. (0.98), бюджет, (0.97), бюджету, (0.95), держбюджет (0.94), бюджету (0.94)"
4,Доменний термін (News/Economics),інвестиції,"інвестувати (0.92), ЄІБ (0.91), онлайн (0.90), більшу (0.90), частку (0.90)","інвестиційні (0.98), інвестицій (0.98), інвестицій. (0.98), інвестиції. (0.98), інвестиційну (0.98)"
5,Морфологічна варіативність (Називний відм.),місто,"літаки (0.94), захопили (0.93), порушили (0.91), правоохоронців (0.91), німецькі (0.91)","міст, (0.93), місті, (0.93), міста, (0.93), місто, (0.93), міста. (0.91)"
6,Морфологічна варіативність (Родовий відм.),міста,"обстрілів (0.87), села (0.86), пункту (0.86), військовослужбовці (0.86), Внаслідок (0.86)","міст (0.92), міста. (0.90), місті (0.88), містах (0.87), місто (0.86)"
7,Варіативність правопису / Новий правопис,проєкт,"проєкту (0.91), внести (0.91), Уряд (0.91), Антикорупційний (0.90), пропозицією (0.90)","проєкту (0.95), проект (0.94), проєкті (0.94), проєкт, (0.93), проєкту. (0.92)"
8,Запозичення / Трансліт,онлайн,"низьку (0.95), нерухомість. (0.94), продукту (0.94), реальних (0.94), «на (0.94)","онлайн, (0.98), онлайн. (0.97), кафе (0.92), электроэнергию, (0.92), City (0.91)"
9,Навмисна опечатка (немає 'и'),презедент,немає в словнику,"інцидент (0.89), презентує (0.88), Інцидент (0.86), премію (0.86), Барак (0.85)"


### Пояснення вибору 10 слів (Відповідно до п. 3.1)

Для перевірки роботи Embeddings на українських текстах ми обрали такі слова:
1. **Часті слова (`держава`, `року`):** Фундамент будь-яких новин/статей. Перевіряємо здатність знаходити базові синоніми (напр. "країна", "місяця").
2. **Доменні терміни (`бюджет`, `інвестиції`):** Перевіряємо кластеризацію економічної та суспільної тематики.
3. **Рідкісне доменне слово (`стартап`):** Вузький термін. Перевіряємо, чи вловив W2V його бізнес-контекст.
4. **Морфологічна варіативність (`місто`, `міста`):** Вкрай важливо для української мови. Перевіряємо, чи W2V знайде семантичних сусідів (селище, столиця), а FastText — просто збережні інші відмінки (містом, місту).
5. **Варіативність правопису (`проєкт`):** Перевіряємо адаптацію моделей до нового українського правопису (сусіди: проект, ініціатива).
6. **Запозичення / Трансліт (`онлайн`):** Перевіряємо, як моделі групують англіцизми в укр. тексті.
7. **Опечатка (`презедент`):** Навмисна помилка. Очікуємо, що Word2Vec видасть OOV (відсутнє в словнику), а FastText завдяки n-грамам виправить ситуацію і знайде "президент".

### Domain terms analysis (5 слів)

In [11]:
domain_terms = ["економіка", "війна", "уряд", "бізнес", "закон"]
domain_results = []

for word in domain_terms:
    w2v_res = get_top_5(w2v_model, word, is_fasttext=False)
    ft_res = get_top_5(ft_model, word, is_fasttext=True)
    
    domain_results.append({
        "Доменний термін": word,
        "Word2Vec (Neighbors)": w2v_res,
        "FastText (Neighbors)": ft_res
    })

df_domain = pd.DataFrame(domain_results)
display(df_domain)

,Доменний термін,Word2Vec (Neighbors),FastText (Neighbors)
0,економіка,"фінансова (0.95), криза (0.95), галузь (0.94), розвиватися (0.94), зростання. (0.94)","економіки. (0.98), економічна (0.97), економіку (0.97), економіку, (0.97), економіці (0.97)"
1,війна,"насправді, (0.96), нікого (0.95), дехто (0.94), розуміє, (0.94), інше (0.94)","жодна (0.90), чітка (0.89), чудова (0.89), вимушена (0.88), війна. (0.88)"
2,уряд,"влада (0.89), Європейський (0.87), завершити (0.87), проєкт (0.85), прийнятий (0.85)","уряд. (0.94), уряд, (0.91), уряди (0.89), Уряд (0.88), верховенство (0.88)"
3,бізнес,"економіку (0.84), бізнесу. (0.84), ринки (0.83), фактор (0.82), проблему (0.82)","бізнес. (0.98), бізнеси (0.98), бізнесі (0.97), бізнес, (0.97), бізнесу? (0.97)"
4,закон,"законопроект (0.86), законопроєкт (0.85), ухвалити (0.82), проект (0.82), закону (0.80)","закон, (0.97), закон. (0.97), законопроєкт, (0.94), закону (0.94), закона (0.94)"


### Висновки до Секції 3.2 (Аналіз доменних термінів)

Аналізуючи 5 ключових доменних термінів корпусу (економіка, війна, уряд, бізнес, закон), ми спостерігаємо дві кардинально різні стратегії роботи моделей через особливості нашого препроцесингу (наявність приклеєної пунктуації):

1. **Word2Vec (Спроба вловити контекст):**
   * **Чи виглядає логічно?** Частково. Модель намагається знаходити слова, що стоять у подібному контексті новин (наприклад, для урядових чи економічних термінів вона підтягує дієслова або пов'язані іменники).
   * **Проблема:** Через те, що пунктуація не була повністю видалена, семантика розмивається. Слово "уряд" і "уряд," для моделі є різними, тому її "сусіди" часто є синтаксичними (ті, що стоять поруч у реченні), а не чистими синонімами.

2. **FastText (Морфологічний збирач):**
   * **Чи виглядає логічно?** З точки зору лексики — абсолютно.
   * **Поведінка:** Замість пошуку семантичних синонімів, FastText знаходить усі відмінки цього ж слова (економіки, економіку) та ігнорує приклеєну пунктуацію (закон., закон,). 

**Яка модель поводиться краще і чому?**
У поточних реаліях нашого корпусу **FastText поводиться набагато стабільніше та корисніше**. 
Українська мова має складну морфологію, і коли на неї накладається шум у вигляді невідділеної пунктуації, словник Word2Vec фрагментується. FastText завдяки n-грамам діє як потужний лематизатор та regex-очищувач "на льоту", групуючи всі варіації доменного терміну в єдиний кластер. Word2Vec міг би виграти лише за умови ідеально чистого тексту та більшого об'єму даних.

## Секція 3.3: 5 кейсів “Корисно / Некорисно”

Цей аналіз демонструє, як Embeddings працюють на реальному україномовному корпусі (без лематизації та з наявністю приклеєної пунктуації).

### Кейс 1: `закон` (КОРИСНО)
* **Word2Vec:** `законопроект`, `законопроєкт`, `ухвалити`, `проект`, `закону`
* **FastText:** `закон,`, `закон.`, `законопроєкт,`, `закону`, `закона`
* **Висновок:** Абсолютно корисно.
* **Чому саме:** Ідеальна синергія. Word2Vec показав блискучий **semantic & functional neighborhood** (знайшов синоніми з різними правописами та пов'язане дієслово "ухвалити"). FastText відпрацював як потужний нормалізатор, зібравши всі морфологічні форми та ігноруючи приклеєну пунктуацію (коми та крапки).

### Кейс 2: `місто` (КОРИСНО — завдяки FastText)
* **Word2Vec:** `літаки`, `захопили`, `порушили`, `правоохоронців`
* **FastText:** `міст,`, `місті,`, `міста,`, `місто,`, `міста.`
* **Висновок:** Корисно (демонструє силу морфології FastText).
* **Чому саме:** Українська мова дуже флективна. Word2Vec тут "зламався" на частотному слові: замість синонімів (селище, столиця) він підтягнув контекстний новинний шум (те, що часто стається в містах у новинах — "захопили", "літаки"). Натомість **morphology helped** у FastText: він ідеально згрупував усі відмінки (Називний, Родовий, Місцевий) та очистив їх від пунктуації.

### Кейс 3: `війна` (НЕКОРИСНО)
* **Word2Vec:** `насправді,`, `нікого`, `дехто`, `розуміє,`, `інше`
* **FastText:** `жодна`, `чітка`, `чудова`, `вимушена`, `війна.`
* **Висновок:** Слабко / Некорисно.
* **Чому саме:** Моделі вивчили синтаксис (syntactic neighborhood) замість семантики. Слово настільки часте в різних контекстах, що Word2Vec просто видає випадкові займенники та стоп-слова, що стоять поруч. FastText видає виключно прикметники, якими це слово описують. Жодна з моделей не видала реальних семантичних синонімів (конфлікт, бойові дії).

### Кейс 4: `презедент` [опечатка] (НЕКОРИСНО)
* **Word2Vec:** `[Немає в словнику] (OOV)`
* **FastText:** `інцидент`, `презентує`, `Інцидент`, `премію`, `Барак`
* **Висновок:** Некорисно (Галюцинації).
* **Чому саме:** Класична проблема noisy data. Word2Vec очікувано і чесно падає через відсутність слова у словнику. Від FastText ми очікували, що його n-грами виправлять опечатку на "президент". Натомість ми отримали **lexical hallucination**: він знайшов слова з подібним набором літер (`інцидент`, `презентує`), які не мають нічого спільного за змістом (хоча ім'я "Барак" є цікавим випадковим збігом контексту).

### Кейс 5: `економіка` (ЧАСТКОВО КОРИСНО / ЗМІШАНИЙ)
* **Word2Vec:** `фінансова`, `криза`, `галузь`, `розвиватися`, `зростання.`
* **FastText:** `економіки.`, `економічна`, `економіку`, `економіку,`, `економіці`
* **Висновок:** Частково корисно (Кожна модель дає лише половину картинки).
* **Чому саме:** W2V показав чудовий **domain neighborhood** — він не знайшов прямих синонімів, але ідеально описав макроекономічний контекст (фінанси, криза, зростання). FT повністю проігнорував контекст і зосередився виключно на морфології (відмінках). Щоб отримати справжню користь, нам довелося б об'єднати результати обох моделей, або ж зробити жорсткий лематизаційний препроцесинг перед навчанням Word2Vec.

## Секція 3.3: 5 кейсів “Корисно / Некорисно”

*(Відповідно до пунктів 4 та 5 методички, ми оцінюємо кейси не за ідеальним збігом синонімів. **"Корисний"** кейс демонструє семантичну чи доменну близькість, або здатність FastText правильно обробити морфологію української мови. **"Некорисний"** кейс виникає там, де особливості частотності слова або шум у даних призводять до випадкових/стилістичних сусідів, або коли FastText генерує лексичні галюцинації на опечатках).*

Цей аналіз демонструє, як Embeddings працюють на реальному загальному україномовному корпусі (без лематизації та з наявністю приклеєної пунктуації).

## Секція 6: Порівняння Word2Vec та FastText

На основі проведених експериментів із загальним україномовним корпусом (UA Datasets, ~965 тис. токенів, вихідні словоформи без лематизації) ми можемо зробити детальний порівняльний аналіз.

### 6.1. На яких словах моделі були приблизно однакові
Обидві моделі показали стабільні результати на **високочастотних загальних словах та добре представлених доменних термінах**, які не мали проблем із правописом.
* Для слів на кшталт `держава` або `року` обидві архітектури змогли захопити правильний суспільно-політичний чи часовий контекст. 
* Проте їхній підхід різниться: там, де обидві моделі "спрацювали", Word2Vec шукав контекстуальних сусідів (дієслова дії, пов'язані іменники), а FastText групував морфологічні варіанти самого слова.

### 6.2. Де FastText був безапеляційно кращим
FastText продемонстрував свою абсолютну перевагу (завдяки підсловним n-грамам) у специфіці української мови та "брудних" даних:
* **Морфологічна варіативність:** Українська мова дуже флективна. FastText ідеально зібрав усі відмінки докупи (Кейс `місто` -> `міст,`, `місті`, `міста`), працюючи як вбудований лематизатор.
* **Noisy forms та приклеєна пунктуація:** FastText блискуче обійшов проблему неідеального препроцесингу. Для нього `закон` і `закон.` — це одне й те саме, тоді як для Word2Vec це два абсолютно різні токени, що розмиває словник.
* **Помилки / Трансліт:** Хоча на слові `презедент` FastText видав шум, він хоча б здатний генерувати вектори для невідомих слів (OOV), відштовхуючись від їхнього буквеного складу, що рятує при аналізі текстів з опечатками.

### 6.3. Де Word2Vec не гірший або навіть простіший для інтерпретації
Word2Vec виявився **семантично чистішим та глибшим** інструментом, коли нам потрібен саме *зміст*, а не граматика.
* **Справжня семантика замість морфології:** Для терміну `економіка` Word2Vec знайшов реальні контекстні зв'язки (`фінансова`, `криза`, `галузь`). Натомість FastText "полінувався" шукати синоніми і просто видав `економіки`, `економіку`, `економіці`. Для задачі пошуку синонімів W2V набагато корисніший.
* **Відсутність галюцинацій (чесність):** Зіткнувшись із опечаткою `презедент`, Word2Vec чесно видав помилку (OOV). FastText же почав шукати слова зі схожими літерами (`інцидент`, `презентує`), створюючи небезпечні "лексичні галюцинації", які можуть зламати логіку downstream-задач.

### 6.4. Підсумковий висновок
**Для нашого поточного корпусу UA Datasets кращою та надійнішою моделлю є FastText.** **Чому?** Корпус не був лематизований і зберіг частину пунктуації. У таких умовах словник Word2Vec катастрофічно фрагментується (окремо існують вектори для "місто", "міста", "містом", "місто,"). FastText завдяки n-грамам діє як потужний "згладжувач" цього шуму і морфології, дозволяючи отримувати стабільні вектори для українських слів у будь-якому відмінку.

Однак, *якби* ми застосували жорсткий препроцесинг (повна лематизація через SpaCy/Stanza та тотальне видалення спецсимволів), **Word2Vec** був би кращим вибором для NLP-задач, оскільки він моделює справжню контекстну семантику (синоніми), а не просто збирає візуально схожі літери.

### Підсумкові таблиці (Секція 7)

In [12]:
# Формуємо таблицю на основі нашого аналізу з Секції 3.3
summary_data = [
    {
        "Word": "закон", 
        "Type": "domain / morph", 
        "Word2Vec neighbors": "законопроект, ухвалити, проект", 
        "FastText neighbors": "закон,, закон., закону", 
        "Useful?": "useful", 
        "Comment": "W2V знайшов синоніми, а FT ідеально очистив пунктуацію."
    },
    {
        "Word": "місто", 
        "Type": "frequent", 
        "Word2Vec neighbors": "літаки, захопили, порушили", 
        "FastText neighbors": "міст,, місті,, міста,", 
        "Useful?": "useful", 
        "Comment": "FT геніально згрупував відмінки (як лематизатор), W2V видав новинний синтаксис."
    },
    {
        "Word": "війна", 
        "Type": "frequent", 
        "Word2Vec neighbors": "насправді,, нікого, розуміє", 
        "FastText neighbors": "жодна, чітка, чудова", 
        "Useful?": "weak", 
        "Comment": "Обидві моделі вивчили синтаксис (слова поруч) замість синонімів."
    },
    {
        "Word": "презедент", 
        "Type": "noisy (typo)", 
        "Word2Vec neighbors": "[OOV]", 
        "FastText neighbors": "інцидент, презентує, Барак", 
        "Useful?": "weak", 
        "Comment": "W2V впав, а FT згенерував лексичні галюцинації замість виправлення."
    },
    {
        "Word": "економіка", 
        "Type": "domain", 
        "Word2Vec neighbors": "фінансова, криза, розвиватися", 
        "FastText neighbors": "економіки., економічна, економіку", 
        "Useful?": "partly", 
        "Comment": "W2V чудово вловив макроконтекст, FT сфокусувався лише на відмінках."
    }
]

df_summary = pd.DataFrame(summary_data)
display(df_summary)

,Word,Type,Word2Vec neighbors,FastText neighbors,Useful?,Comment
0,закон,domain / morph,"законопроект, ухвалити, проект","закон,, закон., закону",useful,"W2V знайшов синоніми, а FT ідеально очистив пунктуацію."
1,місто,frequent,"літаки, захопили, порушили","міст,, місті,, міста,",useful,"FT геніально згрупував відмінки (як лематизатор), W2V видав новинний синтаксис."
2,війна,frequent,"насправді,, нікого, розуміє","жодна, чітка, чудова",weak,Обидві моделі вивчили синтаксис (слова поруч) замість синонімів.
3,презедент,noisy (typo),[OOV],"інцидент, презентує, Барак",weak,"W2V впав, а FT згенерував лексичні галюцинації замість виправлення."
4,економіка,domain,"фінансова, криза, розвиватися","економіки., економічна, економіку",partly,"W2V чудово вловив макроконтекст, FT сфокусувався лише на відмінках."


### 5 головних висновків по корпусу UA Datasets

| № | Висновок | Опис проблеми / Інсайту на нашому корпусі |
|---|---|---|
| **1** | **Флективність української мови** | Без попередньої лематизації Word2Vec розпорошує контекст слова по десятках його відмінків (місто, міста, містом). FastText успішно нівелює цю проблему завдяки n-грамам, працюючи як псевдо-лематизатор. |
| **2** | **Чутливість до приклеєної пунктуації** | Токенізація через звичайний `split()` залишила розділові знаки (`закон.`, `уряд,`). Це роздуло словник Word2Vec шумом. FastText легко ігнорує цей шум і знаходить корінь слова. |
| **3** | **Синтаксис проти Семантики** | Для дуже частих загальних слів (напр., `війна`) моделі схильні видавати займенники, дієслова та прикметники, що найчастіше стоять поруч у тексті новин, а не класичні семантичні синоніми. |
| **4** | **Небезпека галюцинацій FastText** | Хоча FastText рятує при опечатках, на зовсім незнайомих словах (навмисна помилка `презедент` або рідкісні абревіатури) він створює вектори зі схожих літер, видаючи абсолютно непов'язані за змістом слова (`інцидент`). |
| **5** | **Вибір моделі залежить від препроцесингу** | Для "сирих" або слабо очищених українських текстів **FastText є обов'язковим**. Якщо ж провести жорстку лематизацію (напр., через Stanza) та видалити всі спецсимволи — **Word2Vec** дасть набагато глибшу смислову (семантичну) кластеризацію. |

In [13]:
os.makedirs('../docs', exist_ok=True) 

audit_summary_content = """# Audit Summary: Lab 9 (Word Embeddings - UA Datasets)

1. **Який корпус і скільки в ньому даних:**
   Аналізувався корпус загальних українських текстів (UA Datasets). Для тренування було взято репрезентативну вибірку з 5000 документів. Після фільтрації коротких текстів (< 20 слів) робочий корпус склав 4939 документів та 965,294 токени. Текст не лематизувався для перевірки роботи моделей з морфологією.

2. **Які моделі натреновано:**
   Натреновано моделі **Word2Vec** та **FastText** (Gensim). Параметри: `vector_size=100`, `window=5`, `min_count=5`, `sg=1` (Skip-gram). Вибір Skip-gram зумовлений його здатністю краще моделювати рідкісні словоформи в українській мові.

3. **Найсильніші приклади nearest neighbors:**
   * **`місто` (FastText)**: Блискуче згрупував усі відмінки (`місті`, `міста`, `міст`) та проігнорував приклеєну пунктуацію, фактично виконавши роль лематизатора.
   * **`закон` (Word2Vec)**: Знайшов семантично близькі терміни (`законопроект`, `проєкт`) та функціональні зв'язки (`ухвалити`), що свідчить про розуміння контексту.

4. **Найслабші приклади nearest neighbors:**
   * **`презедент` (опечатка)**: Word2Vec видав OOV (чесно визнав відсутність слова), а FastText згенерував лексичні галюцинації (`інцидент`, `презентує`), які не мають семантичного зв'язку, лише схожі літери.
   * **`війна`**: Через високу частотність та різноманітність контекстів обидві моделі видали синтаксичний шум (займенники, прикметники) замість семантичних синонімів.

5. **Які доменні терміни виявилися осмисленими:**
   Термін `економіка` виявився дуже осмисленим у Word2Vec. Модель підтягнула макроекономічний контекст (`криза`, `фінансова`, `галузь`), що є корисним сигналом для тематичного аналізу текстів.

6. **Де FastText виграв:**
   FastText став абсолютним переможцем у роботі з "брудними" даними та морфологією. Завдяки n-грамам він об'єднав словоформи з приклеєною пунктуацією (`закон.`, `бізнес,`) в єдині кластери, з чим Word2Vec не впорався.

7. **Де виграшу майже не було:**
   На дуже частих загальних словах обидві моделі продемонстрували схильність до синтаксичного сусідства (слова, що просто стоять поруч) замість глибокої семантичної близькості. Також FastText не зміг адекватно "виправити" опечатки, замінивши їх випадковими схожими словами.

8. **Чи embeddings варті подальшого використання у вашому кейсі:**
   **Так.** Для української мови FastText є критично важливим інструментом, якщо дані не проходять ідеальну лематизацію. Він дозволяє автоматизувати групування відмінків та нівелювати шум токенізації. Проте для глибокого аналізу смислів рекомендується поєднувати FastText із попередньою лематизацією тексту.
"""

with open("../docs/audit_summary_lab9.md", "w", encoding="utf-8") as f:
    f.write(audit_summary_content)
print("Файл docs/audit_summary_lab9.md успішно згенеровано!")

Файл docs/audit_summary_lab9.md успішно згенеровано!
